In [ ]:
import os
#!/usr/bin/env python
import torch, sys, ast
import numpy as np
import matplotlib.pyplot as plt
# define paths
bomi_ws_path = os.environ.get('BOMI_WS', os.path.abspath('../../..') + '/')  # run from src/bomi-control/scripts or set BOMI_WS
bomicontrol_path = bomi_ws_path +"src/bomi-control"
sys.path.append(bomicontrol_path)

from emg_regression.utils.data_processing import Data_processing, combine_datasets
from cfg.tools import load_yaml, get_subj_day_folder

from emg_regression.utils.torch_helper import TorchHelper
from torch.utils.data import TensorDataset, DataLoader
from emg_regression.approximators.lstm import LSTM
from emg_regression.utils.tools import *
from cfg.tools import get_subj_day_folder
import seaborn as sns
%matplotlib widget

In [ ]:
def process_data(bomicontrol_path,data_path,save_path,crop_duration=7):
    # Load params
    params     = load_yaml(f"{bomicontrol_path}/cfg/params.yaml")
    emg_params = load_yaml(f"{bomicontrol_path}/cfg/emg.yaml")
    data_processing = Data_processing(data_path, emg_params,
                                    fs_recording=params['record']['freq'],
                                    degrees=False, 
                                    downsample_factor=1, 
                                    debug=1)
    data = data_processing.run()
    # Remove last seconds of training
    crop_duration = 7 # seconds -1s before start task and 6 seconds of movement
    nb_samples_train = int(crop_duration * data_processing.fs_features)
    data = data[:,:nb_samples_train,:] # get just first 5 seconds
    data[:,:,0] = data[:,:,0] - data[:,0,0][:,np.newaxis] # all trials starting from 0 time

    # Save dataset
    data_processing.dataset = data.transpose(1,0,2)
    data_processing.save(save_path)

### 1. Processing

In [ ]:
subj = 'S4'
date = '09_04_2024'

######################################################

subj_day_folder = get_subj_day_folder(subj,date)
record_path = f"{subj_day_folder}training/imu/"
train_files = load_yaml(f"{subj_day_folder}files.yaml")['training']
save_folder = f"{subj_day_folder}training/"
# print(data_path, save_path)

for file_name in train_files:
    data_path = f"{record_path}data_{file_name}.pkl" #raw data
    save_path = f"{record_path}imuemg_{file_name}"   #processed data    
    process_data(bomicontrol_path,data_path,save_path,crop_duration=7) # save them

# Combine processed data (7 seconds each)
alldata = combine_datasets(record_path,files=train_files)

# remove baseline EMG:
# alldata[:,:,5:9] = alldata[:,:,5:9] - alldata[:bl_samples,:,5:9].mean(0)[np.newaxis]

# check the data
fs = 100
a, b = fs, fs
fig, ax = plt.subplots(1,2,figsize=(15,7))
for k in range(alldata.shape[1]):
    ax[0].scatter(alldata[:,k,1],alldata[:,k,2],s=2)
    ax[1].scatter(alldata[a:-b,k,1],alldata[a:-b,k,2],s=2)
plt.show()

print("Dataset points:", alldata.shape[0]*alldata.shape[1])

# check emg data
nb_traj = alldata.shape[1]
num_cols = min(nb_traj, 8)  # Maximum of 8 rows
num_rows = (nb_traj+num_cols-1) // num_cols  # Calculate the number of columns
fig, axes = plt.subplots(num_rows, num_cols, figsize=(13,8))  # Adjust figsize as neede
for i,ax in enumerate(axes.flatten()):
    if i < nb_traj:
        ax.plot(alldata[:,i,0],alldata[:,i,5:9])
plt.tight_layout()
plt.show()

print("Final dataset:",alldata.shape)

In [ ]:
figs_path = bomi_ws_path + "figs/"

In [ ]:
alldata.shape # (N, nb_traj, variables)
#columns: [t, lat_pos, front_pos, lat_vel, front_vel, ch1_rms, ch2_rms, ch3_rms, ch4_rms, ... (nzc, 1-4)]
fontsize=12
imu_colors = ["#BCC255", "#FBC591","#F2502F","#7B1E1A"]
imu_labels = [r'$\alpha$',r'$\dot{\alpha}$',r'$\beta$',r'$\dot{\beta}$']
emg_colors = sns.color_palette("viridis", n_colors=4)
emg_labels = ['ch1','ch2','ch3','ch4']

# 30 = go diagonal left, # 5 is going back
for i in range(50):
    fig, ax = plt.subplots(1,3,figsize=(8,3))

    print(i)
    trial_idx = i
    t = alldata[:,trial_idx,0]
    alpha, beta = alldata[:,trial_idx,1], alldata[:,trial_idx,2]
    alpha_dot, beta_dot = alldata[:,trial_idx,3], alldata[:,trial_idx,4]
    imu_vals = [alpha, alpha_dot, beta, beta_dot]
    for i in range(len(imu_vals)):
        ax[0].plot(t,imu_vals[i]*180/np.pi,color=imu_colors[i],lw=4,label=imu_labels[i])
    ax[0].set_xlim([-0.1,5]); 
    ax[0].set_ylim([-25,25]); ax[0].axis('off') 
    ax[0].legend(loc='upper left', bbox_to_anchor=(-0.02,1.2),fontsize=fontsize+7,handletextpad=0.4,handlelength=1.2)

    emg_rms = alldata[:,trial_idx,5:9]

    # emg_labels = ['Right ES','Left ES','Right EO','Left EO']
    # fig, ax = plt.subplots(figsize=(4,3))
    for i in range(len(emg_labels)):
        ax[1].plot(t,emg_rms[:,i],color=emg_colors[i],lw=4,label=emg_labels[i])
    ax[1].legend(loc='upper left', bbox_to_anchor=(-0.02,1.2),fontsize=fontsize+7, handletextpad=0.4,handlelength=1.2)
    ax[1].axis('off') 
    # fig.tight_layout()
    ax[2].scatter(alpha*180/np.pi,beta*180/np.pi,color='k',s=0.2,alpha=0.1)
    ax[2].set_xlim([-20,20])
    ax[2].set_ylim([-40,40])
    plt.tight_layout()
    plt.show()


In [ ]:
alldata.shape # (N, nb_traj, variables)
#columns: [t, lat_pos, front_pos, lat_vel, front_vel, ch1_rms, ch2_rms, ch3_rms, ch4_rms, ... (nzc, 1-4)]
fontsize=12

# 30 = go diagonal left, # 5 is going back
trial_idx = 27
t = alldata[:,trial_idx,0]
alpha, beta = alldata[:,trial_idx,1], alldata[:,trial_idx,2]
alpha_dot, beta_dot = alldata[:,trial_idx,3], alldata[:,trial_idx,4]
imu_colors = ["#BCC255", "#FBC591","#F2502F","#7B1E1A"]
imu_labels = [r'$\alpha$',r'$\dot{\alpha}$',r'$\beta$',r'$\dot{\beta}$']

fig, ax = plt.subplots(figsize=(4,3))
imu_vals = [alpha, alpha_dot, beta, beta_dot]
for i in range(len(imu_vals)):
    ax.plot(t,imu_vals[i]*180/np.pi,color=imu_colors[i],lw=4,label=imu_labels[i])
# ax.set_title("Trunk angular position (°) and velocity (°/s)",fontsize=fontsize,pad=12)
# ax.set_xlabel("Time (s)", fontsize=fontsize, labelpad=5)
ax.set_xlim([-0.1,5]); 
ax.set_ylim([-25,10]); ax.axis('off') 
# ax.legend(loc='upper left', bbox_to_anchor=(-0.02,0.68),fontsize=fontsize+7,handletextpad=0.4,handlelength=1.2)
ax.legend(loc='upper left', bbox_to_anchor=(-0.4,1.1),fontsize=fontsize+7,handletextpad=0.4,handlelength=1.2)
fig.tight_layout()
# plt.savefig(f"{figs_path}imu_data_eg1.png",format='png', dpi=300, bbox_inches='tight', facecolor='w')

emg_rms = alldata[:,trial_idx,5:9]
# emg_rms = emg_rms - emg_rms[0,:]
emg_colors = sns.color_palette("viridis", n_colors=4)
emg_labels = ['ch1','ch2','ch3','ch4']
# emg_labels = ['Right ES','Left ES','Right EO','Left EO']
fig, ax = plt.subplots(figsize=(4,3))
for i in range(len(emg_labels)):
    ax.plot(t,emg_rms[:,i],color=emg_colors[i],lw=4,label=emg_labels[i])
ax.legend(loc='upper left', bbox_to_anchor=(-0.08,1.2),fontsize=fontsize+7, handletextpad=0.4,handlelength=1.2)
ax.axis('off') 
fig.tight_layout()
# plt.savefig(f"{figs_path}emg_data_eg1.png",format='png', dpi=300, bbox_inches='tight', facecolor='w')

In [ ]:
fig, ax = plt.subplots(figsize=(4,3))
N=500
ax.scatter(alpha[:N]*180/np.pi,beta[:N]*180/np.pi,color='b',s=5,alpha=0.9,zorder=3)
xlim, ylim = 25,30
ax.set_xlim([-xlim,xlim]); ax.set_ylim([-ylim,ylim])
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)  # remove ticks and labels
# for spine in ax.spines.values(): spine.set_linewidth(8);spine.set_color('k')
start_circle = plt.Circle((alpha[0]*180/np.pi, beta[0]*180/np.pi), radius=1, edgecolor='blue', facecolor='blue', alpha=0.2, linewidth=1)
end_circle = plt.Circle((alpha[N-1]*180/np.pi, beta[N-1]*180/np.pi), radius=1, edgecolor='blue', facecolor='blue', alpha=0.2, linewidth=1)
ax.add_patch(start_circle)
ax.add_patch(end_circle)
# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)
ax.annotate('', xy=(xlim-1, 0), xytext=(0,0),arrowprops=dict(arrowstyle='->',color='k',alpha=1,linewidth=2))
ax.annotate('', xy=(0, ylim-1), xytext=(0,0),arrowprops=dict(arrowstyle='->',color='k',alpha=1,linewidth=2))
ax.text(xlim-1.5,-2, r'$\alpha$', fontsize=fontsize+9, ha='right', va='top')
ax.text(-1.5,ylim-3, r'$\beta$', fontsize=fontsize+9, ha='right', va='top')
ax.plot(0, 0, 'ko', markersize=3, zorder=5,alpha=0.8)  
for spine in ax.spines.values(): spine.set_linewidth(5) 
# ax.grid(1,alpha=0.2)
ax.axis('off') 
fig.tight_layout()
plt.savefig(f"{figs_path}traj_data_eg1.png",format='png', dpi=300, bbox_inches='tight', facecolor='w')



In [ ]:
trial_idx = 29
emg_rms = alldata[:,trial_idx,5:9]
alpha, beta = alldata[:,trial_idx,1], alldata[:,trial_idx,2]
alpha_dot, beta_dot = alldata[:,trial_idx,3], alldata[:,trial_idx,4]
fig, ax = plt.subplots(figsize=(4,3))
for i in range(len(emg_labels)):
    ax.plot(t,emg_rms[:,i],color=emg_colors[i],lw=4,label=emg_labels[i])
# ax.legend(loc='upper left', bbox_to_anchor=(-0.08,1.2),fontsize=fontsize+7, handletextpad=0.4,handlelength=1.2)
ax.axis('off') 
fig.tight_layout()
plt.savefig(f"{figs_path}emg_target_eg1.png",format='png', dpi=300, bbox_inches='tight', facecolor='w')

fig, ax = plt.subplots(figsize=(4,3))
N=500
ax.scatter(alpha[:N]*180/np.pi,beta[:N]*180/np.pi,color='b',s=5,alpha=0.9,zorder=3)
xlim, ylim = 25,30
ax.set_xlim([-xlim,xlim]); ax.set_ylim([-ylim,ylim])
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)  # remove ticks and labels
# for spine in ax.spines.values(): spine.set_linewidth(8);spine.set_color('k')
start_circle = plt.Circle((alpha[0]*180/np.pi, beta[0]*180/np.pi), radius=1, edgecolor='blue', facecolor='blue', alpha=0.2, linewidth=1)
end_circle = plt.Circle((alpha[N-1]*180/np.pi, beta[N-1]*180/np.pi), radius=1, edgecolor='blue', facecolor='blue', alpha=0.2, linewidth=1)
ax.add_patch(start_circle)
ax.add_patch(end_circle)
for spine in ax.spines.values(): spine.set_linewidth(5) 
# ax.grid(1,alpha=0.2)
ax.axis('off') 
fig.tight_layout()
plt.savefig(f"{figs_path}traj_target_eg1.png",format='png', dpi=300, bbox_inches='tight', facecolor='w')



In [ ]:
# example of training data
lim = 32
y = alldata[:,:,1:]*180/np.pi
fig,ax = plt.subplots(figsize=(6,5))
colors = sns.color_palette('hls', 1000)[::15]
xlabel, ylabel = r'$\alpha$',r'$\beta$'
# xlabel, ylabel = r'$\alpha$ (°)',r'$\beta$ (°)'
xlabel, ylabel = r'Lateral flexion ($\alpha$, °)',r'Frontal flexion ($\beta$, °)'

for i in range(y.shape[1]):
    ax.plot(y[:,i,0], y[:,i,1], color=colors[i], linewidth=2, linestyle='-')
    ax.scatter(y[-1,i,0], y[-1,i,1], color=colors[i])
    ax.scatter(y[0,i,0], y[0,i,1], color='k',alpha=0.5)
ax.set_xlim([-lim,lim])
ax.set_ylim([-lim,lim])
ax.set_xlabel(xlabel,fontsize=fontsize+7,labelpad=9)       
ax.set_ylabel(ylabel,fontsize=fontsize+7,labelpad=5)  
ax.tick_params(axis='both',labelsize=fontsize+3)
fig.tight_layout()
# plt.savefig(f"{figs_path}traindata_eg1.png",format='png', dpi=300, bbox_inches='tight', facecolor='w')

# ax.grid()

### 2. Train

In [ ]:
# set torch device
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load params
params = load_yaml(f"{bomicontrol_path}/cfg/model_params.yaml")
model_path = f"{subj_day_folder}model/{params['model']['name']}"
print_params(params)
print(model_path)
fs = int(1/params['time_step'])

# crop processed data to get from 0 to 5
start_cut, end_cut = fs, -fs
data_loaded = alldata[start_cut:end_cut,:,:]

t    = data_loaded[:,0,0] - data_loaded[0,0,0]
data = data_loaded[:,:,1:]
data, input_dim, output_dim = select_IO(params, data)

# Select input and output data for model
total_nb_trajectories = data.shape[1]
nb_ch = input_dim - output_dim
print('input_dim:', input_dim, ', output_dim:', output_dim)
print('data:', data.shape)
print('Total nb_trajectories:', total_nb_trajectories)

# Check data
check_data(t,data,output_dim,N=5)

In [ ]:
idx_sort = select_test_CMW(data, t, output_dim, nb_options=5)

In [ ]:
# params['train']['split'] = 'cmw'
if params['train']['split'] == 'cmw':
    # idx_sort = select_test_CMW(data, t, output_dim, nb_options=5)
    request = "Select the trajectory indexes for each channel (default: [0,0,0,0]): "
    user_input = input(request)
    idx_selected = [0,0,0,0] if not user_input else ast.literal_eval(user_input)
    print(idx_selected)

    test_idx = [idx_sort[idx_selected[ch],ch] for ch in range(nb_ch)]
    train_idx = list(set(list(range(total_nb_trajectories))) - set(test_idx))
    train_data_raw, test_data_raw = data[:,train_idx,:], data[:,test_idx,:]

    plot_train_test(train_data_raw, test_data_raw,lim=0.8)
    question = "Do you want to proceed with this split? (y/n): "
    proceed = input(question).lower()
    while proceed != 'y':
        request = "Select the trajectory indexes for each channel (default: [0,0,0,0]): "
        idx_selected = ast.literal_eval(input(request))
        test_idx = [idx_sort[idx_selected[ch],ch] for ch in range(nb_ch)]
        train_idx = list(set(list(range(total_nb_trajectories))) - set(test_idx))
        train_data_raw, test_data_raw = data[:,train_idx,:], data[:,test_idx,:]
        plot_train_test(train_data_raw, test_data_raw,lim=0.8)
        proceed = input(question).lower()
        # plt.close('all')
    # save before normalizing
    save_train_test(save_folder,train_data_raw, test_data_raw,train_idx,test_idx)

else:
    # Load previous split
    _, _, train_idx, test_idx = load_train_test(save_folder)
    train_data_raw, test_data_raw = data[:,train_idx,:], data[:,test_idx,:]

# plot_train_test(train_data_raw, test_data_raw,lim=0.8)

In [ ]:
train_data, test_data = train_data_raw, test_data_raw
NTrain, NTest = train_data.shape[1], test_data.shape[1]
print(f"NTrain: {NTrain}, NTest: {NTest}")

# Normalization of trunk ROM
min_theta, max_theta = abs(train_data[:,:,0].min()), abs(train_data[:,:,0].max())
min_phi,   max_phi   = abs(train_data[:,:,1].min()), abs(train_data[:,:,1].max())
max_side = max(min_theta,max_theta)
H_rad = np.array([[max_side,max_side],
                  [min_phi,  max_phi]])
H_deg = H_rad*180/np.pi
print(H_deg)

# Normalize data for model training and save
train_data_norm = normalize_data(train_data, output_dim, 
                      params['train']['normalize_input'], 
                      params['train']['normalize_output'],
                      save_folder, save=True, load=False)

test_data_norm = normalize_data(test_data, output_dim, 
                      params['train']['normalize_input'], 
                      params['train']['normalize_output'],
                      save_folder, save=False, load=True)

# Set input and output for specific LSTM
input_train  = train_data_norm if params['model']['autoregressive'] else train_data_norm[:,:,output_dim:]
output_train = train_data_norm[:,:,:output_dim]
input_test  = test_data_norm if params['model']['autoregressive'] else test_data_norm[:,:,output_dim:]
output_test = test_data_norm[:,:,:output_dim]
# Window data for LSTM
delay_samples = 1 if params['model']['autoregressive'] else 0

## Prepare data for training with 2 datasets
plt.close('all')
window_size1 = 50
window_size2 = 100

train_x1, train_y1 = preprocess_data_for_lstm(
                         input_train, output_train, 
                         window_size1, params['window_step'], 
                         delay_samples, device)
train_x2, train_y2 = preprocess_data_for_lstm(
                         input_train, output_train, 
                         window_size2, params['window_step'], 
                         delay_samples, device)

# Batch loader 1
batch_size  = len(train_x1) if params['train']['batch_size'] == 0 else params['train']['batch_size']
dataset1     = TensorDataset(train_x1, train_y1)
data_loader1 = DataLoader(dataset1, batch_size=batch_size, shuffle=params['train']['shuffle'])
# Batch loader 2
dataset2     = TensorDataset(train_x2, train_y2)
data_loader2 = DataLoader(dataset2, batch_size=batch_size, shuffle=params['train']['shuffle'])


In [ ]:
plt.close('all')
params = load_yaml(f"{bomicontrol_path}/cfg/model_params.yaml")
model_path = f"{subj_day_folder}model/{params['model']['name']}"
# Model
model = LSTM(input_size=input_dim, 
                hidden_dim=params['model']['hidden_dim'], 
                pre_output_size=params['model']['preoutput_size'],
                output_size=output_dim, 
                dropout=params['model']['dropout'],
                n_layers=params['model']['num_layers']).to(device)
print(model_path)

# Load previous model
if params['train']['load']:
    TorchHelper.load(model, model_path, device)
model.train()

# Train model
# model, loss_log, epochs, training_time = train_model(model, params, data_loader1)
# plt.plot(loss_log); plt.show()
# print("Training time: ", training_time)

# Train model
model, loss_log, epochs, training_time = train_model(model, params, data_loader2)
plt.plot(loss_log); plt.show()
print("Training time: ", training_time)

In [ ]:
# save model
TorchHelper.save(model, model_path)
print('Model saved to:',model_path)

# Save training parameters
params_dict = get_params_dict(params,input_dim,output_dim,delay_samples,NTrain)
save_to_yaml(f"{subj_day_folder}model/{params['model']['name']}_config.yaml", params_dict)

### 3. Test

In [ ]:
plt.close('all')
params = load_yaml(f"{bomicontrol_path}/cfg/model_params.yaml")
model_path = f"{subj_day_folder}model/{params['model']['name']}"
# Model
model = LSTM(input_size=input_dim, 
                hidden_dim=params['model']['hidden_dim'], 
                pre_output_size=params['model']['preoutput_size'],
                output_size=output_dim, 
                dropout=params['model']['dropout'],
                n_layers=params['model']['num_layers']).to(device)
print(model_path)

# Load previous model
TorchHelper.load(model, model_path, device)

In [ ]:
print('Model loaded:',model_path)
model.eval()

SAVE = 0
window_size = 100

# Training set prediction
ypred_train_ol = predict_motion(train_data_norm,model,output_dim,window_size,device,autoregress=False)
ypred_train_cl = predict_motion(train_data_norm,model,output_dim,window_size,device,autoregress=True)
# Testing set
ypred_ol_test = predict_motion(test_data_norm,model,output_dim,window_size,device,autoregress=False)
ypred_cl_test = predict_motion(test_data_norm,model,output_dim,window_size,device,autoregress=True)

# MSE train
ylabels = [r'$\theta$ (rad)', r'$\phi$ (rad)', r'$\dot{\theta}$ (rad/s)', r'$\dot{\phi}$ (rad/s)']
ytrain = train_data_norm[:,:,:output_dim].transpose(1,0,2)
mse_ol, mse_cl = get_mse(ytrain,ypred_train_ol), get_mse(ytrain,ypred_train_cl)
ave_mse_ol, ave_mse_cl = mse_ol.mean(), mse_cl.mean()

# MSE test
ytest = test_data_norm[:,:,:output_dim].transpose(1,0,2)
mse_test_ol, mse_test_cl = get_mse(ytest,ypred_ol_test), get_mse(ytest,ypred_cl_test)
ave_mse_test_ol, ave_mse_test_cl = mse_test_ol.mean(), mse_test_cl.mean()

#colors
c_train, c_test = sns.color_palette('hls', 500)[::10], sns.color_palette('hls', 100)[::20]

fig, ax = plt.subplots(output_dim,2,figsize=(8,8))
for j in range(output_dim):
    for traj_id in range(NTrain):
        ax[j,0].plot(t,mse_ol[traj_id,:,j],'-o',color=c_train[traj_id],markersize=1,alpha=0.5,label=f'Traj {traj_id+1}')
    for traj_id in range(NTest):
        ax[j,1].plot(t,mse_test_ol[traj_id,:,j],'-o',color=c_test[traj_id],markersize=1,alpha=0.5,label=f'Traj {traj_id+1}')
    ax[j,0].plot(t,mse_ol.mean(0)[:,j],'-o',color='k',markersize=0.5)
    ax[j,1].plot(t,mse_test_ol.mean(0)[:,j],'-o',color='k',markersize=0.5)
    ax[j,0].set_ylabel(ylabels[j])
    ax[j,0].grid(); ax[j,1].grid()
ax[0,0].set_title('MSE Training'); ax[-1,0].set_xlabel('Time (s)')
ax[0,1].set_title('MSE Testing'); ax[-1,1].set_xlabel('Time (s)')
plt.tight_layout()
if SAVE:
    plt.savefig(f"{subj_day_folder}figs/mse_traj_{params['model']['name']}.png",format='png', dpi=300, bbox_inches='tight', facecolor='w')
plt.show()


# plot trajectories (normalized angles)
fig, axes = plt.subplots(2,2,figsize=(10,8))
plot_2Dtraj(axes[0,0], ytrain, ypred_train_ol, title=f'Open-loop Training prediction (N={NTrain}, MSE={ave_mse_ol.round(5)})',   colors=c_train)
plot_2Dtraj(axes[0,1], ytrain, ypred_train_cl,  title=f'Closed-loop Training prediction (N={NTrain})',colors=c_train)
plot_2Dtraj(axes[1,0], ytest, ypred_ol_test, title=f'Open-loop Testing prediction (N={NTest}, MSE={ave_mse_test_ol.round(5)})',  colors=c_test)
plot_2Dtraj(axes[1,1], ytest, ypred_cl_test, title=f'Closed-loop Testing prediction (N={NTest})',colors=c_test)
plt.tight_layout()
if SAVE:
    plt.savefig(f"{subj_day_folder}figs/traj_pred_{params['model']['name']}.png",format='png', dpi=300, bbox_inches='tight', facecolor='w')
plt.show()


""" Map predicted trajectories to screen (measured and predicted)"""

# First, get the predicted trunk angles in radians
mu_y  = np.load(f"{save_folder}/mu_y.npy")
std_y = np.load(f"{save_folder}/std_y.npy")
ytrue_test_deg = ((ytest * std_y) + mu_y)*180/np.pi
# ypred_test_deg = ((ypred_ol_test * std_y) + mu_y)*180/np.pi
ypred_test_deg = ((ypred_cl_test * std_y) + mu_y)*180/np.pi

# Map 
H_imu_deg = np.load(f'{subj_day_folder}interface/H_imu_deg.npy')
d1, d2 =   np.load(f'{subj_day_folder}interface/screen.npy') # must be run in the same device of gui
margin = 0
traj_true_screen = np.array([map_pred_to_screen(traj,d1,d2,H_imu_deg,home_loc='center',margin=margin) 
                             for traj in ytrue_test_deg])
traj_pred_screen = np.array([map_pred_to_screen(traj,d1,d2,H_imu_deg,home_loc='center',margin=margin) 
                             for traj in ypred_test_deg])

# Save reference EMG and IMU NOT normalized
ref_emg = test_data[:,:,output_dim:].transpose(1,0,2)
ref_imu = test_data[:,:,:output_dim].transpose(1,0,2)

plot_reference(y=traj_pred_screen, ytrue=traj_true_screen, 
               t=t-t[0], ref_emg=ref_emg, 
               xlim=[0,d1],ylim=[d2,0],ylim_emg=None)
if SAVE:
    plt.savefig(f"{subj_day_folder}figs/reference_test_MVCnorm.png",format='png', dpi=300, bbox_inches='tight', facecolor='w')
plt.show()

# # Things to save
if SAVE:
    np.save(f"{subj_day_folder}testing/ref_emg", ref_emg)
    np.save(f"{subj_day_folder}testing/ref_imu", ref_imu)
    print("Saved: ref_emg, ref_imu")
    print("Saved: ref_emg_norm, ref_imu_norm")

    # Interface
    np.save(f'{subj_day_folder}interface/H_imu_deg',H_deg)
    np.save(f'{subj_day_folder}interface/H_imu_rad',H_rad)
    np.save(f'{subj_day_folder}interface/screen',(d1,d2))
    np.save(f'{subj_day_folder}interface/traj_screen_true',traj_true_screen)
    np.save(f'{subj_day_folder}interface/traj_screen_pred',traj_pred_screen)
    np.save(f'{subj_day_folder}interface/ytrue_test_deg',ytrue_test_deg)
    np.save(f'{subj_day_folder}interface/ypred_test_deg',ypred_test_deg)
    print("Saved: screen_dim, traj_screen true and pred")

# EMG before normalizing
# emg_mvc = train_data[:,:,output_dim:].max((0,1))
# np.save(emg_mvc_path,emg_mvc)
# print(emg_mvc)
